# T13 — RAG-Agent

T12 hat den Retrieval-Ablauf von Hand ausgeführt. Hier kommt `RetrieveRelevantChunks` in einen `ToolAgent`, der den Ablauf übernimmt und die `_sources` aller Tool-Calls in `answer.sources` sammelt. Das Ergebnis ist ein RAG-Agent, der nur sucht, wenn es nötig ist.

Code, Prompts und Dokumente sind auf Englisch (Begründung in T01).

## Setup

Zuerst T01 ausführen: Dort entsteht der Vector Store unter `backend/db/data_vs.db`. Ist der Store leer, baut `load_or_build_vector_store()` ihn aus allen Dokumenten auf; das dauert einige Minuten.

In [ ]:
import sys

from loguru import logger

from conversational_toolkit.agents.tool_agent import ToolAgent
from conversational_toolkit.llms.base import LLMMessage, MessageContent, Roles
from conversational_toolkit.llms.openai import OpenAILLM
from conversational_toolkit.retriever.vectorstore_retriever import VectorStoreRetriever

from ki_kompass_rag.baseline_rag import RETRIEVER_TOP_K, ask, load_or_build_vector_store
from ki_kompass_rag.tool_agents import RetrieveRelevantChunks

logger.remove()
logger.add(sys.stderr, level="WARNING")

vector_store, embedding_model = await load_or_build_vector_store()
retriever = VectorStoreRetriever(embedding_model, vector_store, top_k=RETRIEVER_TOP_K)

## Der Agent

`RAG_PROMPT` übersetzt:

> Du bist ein Assistent der PrimePack AG. Antworte kurz. Rufe retrieve_relevant_chunks bei Fragen zu Produkten, Lieferanten oder Dokumenten von PrimePack auf. Wenn du das tust, stütze dich nur auf die gefundenen Chunks und nenne ihre Quelldateien.

In [ ]:
RAG_PROMPT = (
    "You are an assistant for PrimePack AG. Answer briefly. "
    "Call retrieve_relevant_chunks for questions about PrimePack products, suppliers "
    "or documents. When you do, rely only on the retrieved chunks and cite their "
    "source files."
)

llm = OpenAILLM(tools=[RetrieveRelevantChunks(retriever)], tool_choice="auto")
rag_agent = ToolAgent(system_prompt=RAG_PROMPT, llm=llm, max_steps=5)

## Konversation

Der Agent speichert die Konversation nicht; `chat` hängt jede Frage und Antwort an `history` an und übergibt den Verlauf mit der nächsten Frage. `ToolAgent` fügt den System-Prompt selbst hinzu, `history` enthält also nur Nachrichten von Nutzer und Assistent.

`Sources: none` heisst, der Agent hat ohne Retrieval geantwortet.

In [ ]:
history: list[LLMMessage] = []


async def chat(agent: ToolAgent, query: str) -> None:
    answer = await ask(agent, query, history)
    user_message = LLMMessage(
        role=Roles.USER, content=[MessageContent(type="text", text=query)]
    )
    history.extend([user_message, answer])

In [ ]:
await chat(
    rag_agent, "What is Einstein's theory of relativity? Answer in 2-3 sentences."
)

In [ ]:
await chat(rag_agent, "Which pallets in our portfolio have a third-party verified EPD?")

Diese Frage braucht nur den Verlauf, kein Retrieval.

In [ ]:
await chat(rag_agent, "Summarize our conversation, one line per message.")